## Aeropulse — Silver: Carrier Dimension

**Purpose:** Bronze → Silver for the carrier reference table. Standardises column names, removes nulls/duplicates on `carrier_code`, and merges into `silver.carrier`.

**Load type:** Full refresh (bronze_carrier is fully overwritten on every ingestion run, so this always processes the complete reference set — no batch-window filtering needed).

**Depends on:** `silver-environment`, `silver-helper` (run via `%run`)

**Reads:** `aeropulse_bronze_lh.dbo.bronze_carrier` (via `carrier_bronze_path`)

**Writes:** `silver.carrier` (merge on `carrier_code`)

**Default lakehouse:** `aeropulse_silver_lh`


In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

StatementMeta(, 99545609-58a7-401f-b242-110754644c36, 4, Finished, Available, Finished, False)

In [3]:
batch_id = ""
batch_year = ""

StatementMeta(, 99545609-58a7-401f-b242-110754644c36, 5, Finished, Available, Finished, False)

In [4]:
%run silver-environment

StatementMeta(, 99545609-58a7-401f-b242-110754644c36, 6, Finished, Available, Finished, True)

In [5]:
%run silver-helper

StatementMeta(, 99545609-58a7-401f-b242-110754644c36, 9, Finished, Available, Finished, True)

In [6]:
carrier_df = spark.read.format('delta').load(carrier_bronze_path).filter(F.col("batch_id") == batch_id)

StatementMeta(, 99545609-58a7-401f-b242-110754644c36, 10, Finished, Available, Finished, False)

In [7]:
# Data cleaning/transformation & standardisation

# dict: for rename mapping
rename_mapping = {
    "Code":"carrier_code",
    "Description":"carrier_description"
}

# function: rename_column headers
carrier_df = (
    carrier_df
    .transform(
        trim_whitespaces)
        .transform(lambda d: rename_column(d, rename_mapping))
)

# remove nulls using remove_nulls function
carrier_df = remove_nulls(carrier_df, ["carrier_code"])


# remove duplicates
carrier_df = remove_duplicates(carrier_df, ["carrier_code"])

# create surrogate key
carrier_df = add_sk_key(carrier_df, ["carrier_code"], "carrier_sk")

StatementMeta(, 99545609-58a7-401f-b242-110754644c36, 11, Finished, Available, Finished, False)

In [8]:
update_cols = [c for c in carrier_df.columns if c not in ["carrier_code"]]

write_to_silver(
    carrier_df,
    "silver.carrier",
    "s.carrier_code = t.carrier_code",
    update_cols
)

StatementMeta(, 99545609-58a7-401f-b242-110754644c36, 12, Finished, Available, Finished, False)